## 규칙기반 AI

In [ ]:
POSITIVE = ["좋", "최고", "만족", "추천", "맛있"]
NEGATIVE = ["별로", "최악", "실망", "불친절", "맛없"]

def classify(text):
    pos = sum(w in text for w in POSITIVE)   # 긍정 단어가 몇 개 들어 있나
    neg = sum(w in text for w in NEGATIVE)   # 부정 단어가 몇 개 들어 있나
    if pos > neg:
        return "긍정"
    if neg > pos:
        return "부정"
    return "모르겠음"

print(classify("음식이 정말 맛있어요"))   # 긍정

긍정


In [2]:
print(classify("음식이 정말 맛있어요"))
print(classify("서비스가 최악이에요"))
print(classify("배송이 빨라서 만족합니다"))
print(classify("직원이 불친절했어요"))
print(classify("생각보다 좋지 않네요"))
print(classify("별로 기대 안 했는데 최고였어요"))
print(classify("가격 대비 나쁘지 않아요"))
print(classify("참 잘~하는 짓이다"))
print(classify("존맛탱 재방문 각"))
print(classify("넘 조아요"))
print(classify("다시는 안 갈 거예요"))
print(classify("맛있다는 사람들 이해가 안 감"))

긍정
부정
긍정
부정
긍정
모르겠음
모르겠음
모르겠음
모르겠음
모르겠음
모르겠음
긍정


#### 머신러닝

In [4]:
train_texts = [
    "정말 맛있고 친절해요", "최고의 식당입니다", "재방문 의사 있어요",
    "가격도 괜찮고 만족해요", "친구에게 추천했어요", "생각보다 훨씬 좋아요",
    "나쁘지 않아요 또 올게요", "존맛 재방문 각",
    "맛이 별로예요", "다시는 안 갈 거예요", "직원이 불친절해요",
    "기대보다 실망스러워요", "좋지 않았어요", "최악의 경험이었어요",
    "돈이 아까워요", "추천하고 싶지 않아요",
]
train_labels = [1, 1, 1, 1, 1, 1, 1, 1,
                0, 0, 0, 0, 0, 0, 0, 0]

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vec = CountVectorizer(ngram_range=(1, 2))   # 단어 1개, 연속된 단어 2개를 센다
X = vec.fit_transform(train_texts)          # 문장 → 단어 빈도 표
model = MultinomialNB().fit(X, train_labels)  # 학습: 단어별 긍정/부정 확률을 센다

tests = ["생각보다 좋지 않네요", "가격 대비 나쁘지 않아요", "다시는 안 갈래요"]
for t, p in zip(tests, model.predict(vec.transform(tests))):
    print(t, "→", "긍정" if p == 1 else "부정")

생각보다 좋지 않네요 → 부정
가격 대비 나쁘지 않아요 → 긍정
다시는 안 갈래요 → 부정


In [6]:
words = vec.get_feature_names_out()
score = model.feature_log_prob_[1] - model.feature_log_prob_[0]   # + 긍정 쪽, - 부정 쪽
ranked = sorted(zip(score, words))
print("부정 신호:", [w for _, w in ranked[:5]])
print("긍정 신호:", [w for _, w in ranked[-5:]])

부정 신호: ['거예요', '경험이었어요', '기대보다', '기대보다 실망스러워요', '다시는']
긍정 신호: ['친구에게 추천했어요', '친절해요', '훨씬', '훨씬 좋아요', '재방문']


#### gpt 발전과정

In [ ]:
#!pip -q install -U "bitsandbytes>=0.46.1" transformers accelerate

In [4]:
import torch, bitsandbytes, math
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
import os 
from dotenv import load_dotenv
from huggingface_hub import login

# .env 파일 로드
load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')
login(token=HF_TOKEN)

In [5]:
print(torch.cuda.is_available(), bitsandbytes.__version__)

True 0.50.2


In [7]:
from transformers import pipeline, set_seed, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [13]:
# 문자열 생성

prompt = "The restaurant was "

# 2018 GPT-1, GPT-2 버전 테스트(베이스 모델)

for name in ['openai-community/openai-gpt','openai-community/gpt2','openai-community/gpt2-xl']:
  set_seed(42)
  gen = pipeline('text-generation', model=name, device=device)

  n = sum(p.numel() for p in gen.model.parameters())
  # 새로 생성되는 토큰 : 최대 40 개,
  out = gen(prompt, max_new_tokens=40,do_sample=True, top_k=50, temperature=0.8)
  print(f"==={name} ({n/1e6:,.0f}M) ===")
  print(out[0]['generated_text'],'\n')
  del gen; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] OpenAIGPTLMHeadModel LOAD REPORT from: openai-community/openai-gpt
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


===openai-community/openai-gpt (117M) ===
The restaurant was deserted . she did n't see anyone lurking behind the table when she went to stand next to him . she looked around for an elevator , but there was none . she walked around to the front of the 



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

===openai-community/gpt2 (124M) ===
The restaurant was  re-opened in 2005, but closed again in 2009 after an investigative investigation by police department of the owner and the alleged fraud. The investigation led to 



model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

... (4 outputs left out)

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

===openai-community/gpt2-xl (1,558M) ===
The restaurant was iced to the ground"

In December 2014, the Supreme Court of Canada dismissed the appeal of a landlord who was forced to evacuate a condominium for a few hours in December 2012 when the roof 



In [14]:
import math

text = ("Last weekend my friends and I tried a new pizza place near the station. "
"The dough was thin and crispy, and the sauce tasted fresh. "
"The service was a little slow because the restaurant was crowded, " "but the waiter apologized and brought us free drinks. "
"We will probably go back next month.")

# gpt2 (gpt2-small), gpt2-medium, gpt2-large, gpt2-xl 모델의 파라미터 수와 loss, perplexity를 계산
for name in ["openai-community/gpt2","openai-community/gpt2-medium","openai-community/gpt2-large","openai-community/gpt2-xl"]:
  tok = AutoTokenizer.from_pretrained(name)
  model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
  ids = tok(text, return_tensors="pt").input_ids.to(device)
  with torch.no_grad():
    loss = model(ids, labels=ids).loss.item()
    # 파라미터 수 확인
    n = sum(p.numel() for p in model.parameters())
    # perplexity : 다음 토큰을 평균 몇 개 후보 중에서 고민하는가?
    # loss : 예측한 결과와 실제 정답의 차이
    print(f"{name:30s} {n/1e6:6,.0f}M loss={loss:.3f} perplexity= {math.exp(loss):.1f}")
    del model; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


openai-community/gpt2             124M loss=3.112 perplexity= 22.5


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

... (15 outputs left out)

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

openai-community/gpt2-large       774M loss=2.502 perplexity= 12.2


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

openai-community/gpt2-xl        1,558M loss=2.478 perplexity= 11.9


#### 인컨텍스트 러닝(in-context learning) : 프롬프트 안에서 배워서 새 작업에 적용
- 프롬프트 : 지시어 => zero-shot
- 프롬프트 : 지시어 + 예시 1 개 => one-shot
- 프롬프트 : 지시어 + 예시 여러 개 => few-shot


In [9]:
#MODEL = "Qwen/Qwen2.5-0.5B(빌리언-10억)"
MODEL = "Qwen/Qwen2.5-1.5B"

if "model" in globals():
  del model
  torch.cuda.empty_cache()

# 모델에 맞는 Tokenizer 불러오기
tok = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(device).eval()
n = sum(p.numel() for p in model.parameters())
print(f"{MODEL} : 파라미터 {n/1e6:6,.0f}M개")

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen/Qwen2.5-1.5B : 파라미터  1,544M개


In [ ]:
text = ("Last weekend my friends and I tried a new pizza place near the station. "
"The dough was thin and crispy, and the sauce tasted fresh. "
"The service was a little slow because the restaurant was crowded, " "but the waiter apologized and brought us free drinks. "
"We will probably go back next month.")

# gpt2 (gpt2-small), gpt2-medium, gpt2-large, gpt2-xl 모델의 파라미터 수와 loss, perplexity를 계산
for name in ["openai-community/gpt2","openai-community/gpt2-medium","openai-community/gpt2-large","openai-community/gpt2-xl"]:
  tok = AutoTokenizer.from_pretrained(name)
  model = AutoModelForCausalLM.from_pretrained(name).to(device).eval()
  ids = tok(text, return_tensors="pt").input_ids.to(device)
  with torch.no_grad():
    loss = model(ids, labels=ids).loss.item()
    # 파라미터 수 확인
    n = sum(p.numel() for p in model.parameters())
    print(f"{name:30s} {n/1e6:6,.0f}M loss={loss:.3f} perplexity= {math.exp(loss):.1f}")
    del model; torch.cuda.empty_cache()

In [7]:
TESTS = [
  ("The food was delicious.", "positive"),
  ("The staff were rude.",    "negative"),  # 부정어
  ("Not bad at all for the price.",  "positive"),
  ("I wouldn't call it good.",  "negative"),  # 부정어
  ("Oh great, another hour of waiting. Just perfect.",  "negative"),  # 반어
  ("Wow, what amazing service. Nobody came for 30 minutes.", "negative"),  # 반어
  ("This burger slaps, no cap.",   "positive"),  # 신조어
  ("The fries were mid.",          "negative"),  # 신조어
  ("Teh servise was amazng!!",     "positive"),  # 오타
  ("Never comming back here agian.","negative"),  # 오타
  ("It was okay, I guess.",         "negative"),  # 애매: ?
  ("Well, considering the price...","positive"),  # 애매: ?
]

# 반어문장
TESTS.append(("The interior was beautiful and the staff were friendly, but the food was honestly the worst.", "negative"))

# zero-shot: 지시만
ZERO_SHOT = "Decide whether each restaurant review is positive or negative.\n\n"

# few-shot: 예시 4개. "Sentiment: "의 띄어쓰기까지 형식을 똑같이 맞춘다
FEW_SHOT = ("Review: The pasta was amazing.\nSentiment: positive\n\n"
         "Review: The waiter ignored us for 20 minutes.\nSentiment: negative\n\n"
         "Review: Pretty decent for what you pay.\nSentiment: positive\n\n"
         "Review: I regret ordering this.\nSentiment: negative\n\n")


@torch.no_grad()
def predict(header, review):
    """프롬프트 끝에 올 다음 토큰이 ' positive'일 확률과 ' negative'일 확률을 비교한다."""
    prompt = header + f"Review: {review}\nSentiment:"
    ids = tok(prompt, return_tensors="pt").to(model.device)
    logits = model(**ids).logits[0, -1].float() # 다음 토큰 후보 전체의 점수
    pos = tok.encode(" positive", add_special_tokens=False)[0]
    neg = tok.encode(" negative", add_special_tokens=False)[0]
    p_pos = torch.softmax(logits[[pos, neg]], dim=0)[0].item()# 둘중 positive의 비중
    return ("positive" if p_pos > 0.5 else "negative"), p_pos

def evaluate(header, title):
    print(f"--- {MODEL} | {title} ---")
    correct = 0
    for review, label in TESTS:
        pred, p = predict(header, review)
        correct += (pred == label)
        mark = "O" if pred == label else "X"
        print(f"{mark}  P(positive)={p:.2f}  {review}")
    print(f"=> {correct}/{len(TESTS)}\n")

In [10]:
evaluate(ZERO_SHOT, "zero-shot")
evaluate(FEW_SHOT, "few-shot")

--- Qwen/Qwen2.5-1.5B | zero-shot ---
O  P(positive)=0.98  The food was delicious.
O  P(positive)=0.00  The staff were rude.
O  P(positive)=0.82  Not bad at all for the price.
O  P(positive)=0.01  I wouldn't call it good.
O  P(positive)=0.00  Oh great, another hour of waiting. Just perfect.
O  P(positive)=0.11  Wow, what amazing service. Nobody came for 30 minutes.
O  P(positive)=0.67  This burger slaps, no cap.
O  P(positive)=0.03  The fries were mid.
O  P(positive)=0.97  Teh servise was amazng!!
O  P(positive)=0.00  Never comming back here agian.
O  P(positive)=0.04  It was okay, I guess.
X  P(positive)=0.07  Well, considering the price...
O  P(positive)=0.01  The interior was beautiful and the staff were friendly, but the food was honestly the worst.
=> 12/13

--- Qwen/Qwen2.5-1.5B | few-shot ---
O  P(positive)=1.00  The food was delicious.
O  P(positive)=0.00  The staff were rude.
O  P(positive)=0.99  Not bad at all for the price.
O  P(positive)=0.01  I wouldn't call it good.
O  P(

In [11]:
FLIPPED = (FEW_SHOT.replace("Sentiment: positive", "Sentiment: TMP")
                   .replace("Sentiment: negative", "Sentiment: positive")
                   .replace("Sentiment: TMP", "Sentiment: negative"))

evaluate(FLIPPED, "few-shot (라벨 뒤집음)")

--- Qwen/Qwen2.5-1.5B | few-shot (라벨 뒤집음) ---
O  P(positive)=0.64  The food was delicious.
O  P(positive)=0.13  The staff were rude.
X  P(positive)=0.44  Not bad at all for the price.
O  P(positive)=0.11  I wouldn't call it good.
O  P(positive)=0.07  Oh great, another hour of waiting. Just perfect.
O  P(positive)=0.32  Wow, what amazing service. Nobody came for 30 minutes.
X  P(positive)=0.50  This burger slaps, no cap.
O  P(positive)=0.24  The fries were mid.
O  P(positive)=0.55  Teh servise was amazng!!
O  P(positive)=0.14  Never comming back here agian.
O  P(positive)=0.21  It was okay, I guess.
X  P(positive)=0.19  Well, considering the price...
O  P(positive)=0.06  The interior was beautiful and the staff were friendly, but the food was honestly the worst.
=> 10/13



In [12]:
q = "What is the capital of France?"

# 베이스 모델: 글을 이어 쓴다
base = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B",device_map="auto", dtype="auto")
for p in [q, f"Q: {q}\nA:"]:
    out = base(p, max_new_tokens=60, do_sample=False)[0]["generated_text"]
    print("[BASE]", repr(out), "\n")
del base; torch.cuda.empty_cache()

# 채팅 모델: 지시 튜닝 + RLHF를 거쳐 대답한다
chat = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct", device_map="auto", dtype="auto")
out = chat([{"role": "user", "content": q}], max_new_tokens=60, do_sample=False)
print("[INSTRUCT]", out[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[BASE] 'What is the capital of France? The capital of France is Paris.' 

[BASE] 'Q: What is the capital of France?\nA: Paris\nYou are an AI assistant that follows instruction extremely well. Help as much as you can.' 



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

... (2 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[INSTRUCT] The capital of France is Paris.


In [13]:
# 상사에게 하루 휴가를 요청하는 짧은 이메일을 작성하세요. 1. 프랑스의 수도는 어디인가요?

base = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B",device_map="auto", dtype="auto")
for p in ["Write a short email to my boss asking for a day off.",
          "1. What is the capital of France?\n"]:
    out = base(p, max_new_tokens=80, do_sample=False)[0]["generated_text"]
    print("[BASE]", repr(out), "\n")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[BASE] "Write a short email to my boss asking for a day off. Subject: Request for a Day Off\n\nDear [Boss's Name],\n\nI hope this email finds you well. I am writing to request a day off from work on [Date]. I have been feeling quite overwhelmed with the workload and have been unable to keep up with all the tasks that have been assigned to me.\n\nI understand that it is important to maintain a work-life balance, and I am grateful" 

[BASE] '1. What is the capital of France?\n2. What is the capital of Germany?\n3. What is the capital of Italy?\n4. What is the capital of Spain?\n5. What is the capital of the United States?\n6. What is the capital of Canada?\n7. What is the capital of Australia?\n8. What is the capital of Japan?\n9. What is the capital of China?\n10. What is the' 

